In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
import string

import torch
import torch.nn as nn
import warnings
warnings.filterwarnings('ignore')

In [2]:
plt.rcParams['font.family']='Arial'

bar_width=0.25
plt.rc('axes', labelsize=10)
plt.rc('xtick', labelsize=10) 
plt.rc('ytick', labelsize=10) 
plt.rc('legend', fontsize=10)
plt.rc('figure', titlesize=12) 
alphsize=12
alpha=0.4
padsize=15
set_dpi=600

# SSP

In [ ]:
data = pd.read_excel('./data/heat_region.xlsx')
data_SSP126 = pd.read_csv('./data/SSP126_data.csv')   
data_SSP245 = pd.read_csv('./data/SSP245_data.csv')
data_SSP370 = pd.read_csv('./data/SSP370_data.csv')
data_SSP585 = pd.read_csv('./data/SSP585_data.csv')

In [4]:
regions_korean = ['서울특별시', '부산광역시', '대구광역시', '인천광역시', '광주광역시', '대전광역시', '울산광역시',
                  '세종특별자치시', '경기도', '강원도', '충청북도', '충청남도', '전라북도', '전라남도', '경상북도',
                  '경상남도', '제주특별자치도']

regions_english = ['Seoul', 'Busan', 'Daegu', 'Incheon', 'Gwangju', 'Daejeon', 'Ulsan',
                   'Sejong', 'Gyeonggi', 'Gangwon', 'Chungbuk', 'Chungnam', 'Jeonbuk', 'Jeonnam', 'Gyeongbuk',
                   'Gyeongnam', 'Jeju']

In [5]:
data=data[data['year']>=2018]
data=data.dropna()
data.reset_index(drop=True, inplace=True)

In [6]:
weight_a = [1.0,0.8,0.6]
data['weight_wbgt']=weight_a[0]*data['wbgt']+weight_a[1]*data['wbgt_1_day_ago']+weight_a[2]*data['wbgt_2_day_ago']

data['COVID19']=0
data.loc[(data['year']>=2020) & (data['year']<=2021),'COVID19']=1

In [7]:
features = ['tmax', 'tavg', 'rhum', 'weight_wbgt','wbgt',
            'metro','sex_rate', 'elder_rate', 'COVID19','month',
        're_wbgt', 're_rate', 'phase', 'cons', 'pop', '60+',
       'agri_area1', 'agri_area2', 'agri_pop', 'simple_job_pop','pop_density']

In [8]:
start_year = 2018
end_year = 2022

data_train = data[(data['year']>=start_year) & (data['year']<end_year)]
data_test = data[data['year']>=end_year]

X = pd.concat([data_train.loc[:, features],data_test.loc[:, features]])
X_max = X.max()
X_min = X.min()

In [9]:
window_size = 3

In [10]:
def make_dataset_X(x_data, window_size):
    x_list = []
    for i in range(len(x_data) - window_size+1):
        x_list.append(np.array(x_data.iloc[i:i+window_size]))
    x_list = np.array(x_list)
    return x_list

In [11]:
data_SSP126['region'] = data_SSP126['region'].replace(regions_korean, regions_english)
data_SSP245['region'] = data_SSP245['region'].replace(regions_korean, regions_english)
data_SSP370['region'] = data_SSP370['region'].replace(regions_korean, regions_english)
data_SSP585['region'] = data_SSP585['region'].replace(regions_korean, regions_english)

In [12]:
data_SSP126.rename(columns={'con':'cons'},inplace=True)
data_SSP245.rename(columns={'con':'cons'},inplace=True)
data_SSP370.rename(columns={'con':'cons'},inplace=True)
data_SSP585.rename(columns={'con':'cons'},inplace=True)

In [13]:
Region=data.region.unique()

In [14]:
import random
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU.
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

In [15]:
class LSTM(nn.Module):
    def __init__(self):
        super(LSTM, self).__init__()

        self.lstm = nn.LSTM(input_size = len(features), hidden_size=8, num_layers=5,
                          batch_first = True)

        self.fc1 = nn.Linear(in_features=window_size*8, out_features=128)
        # self.fc1 = nn.Linear(in_features=window_size*8, out_features=64)
        self.fc2 = nn.Linear(in_features=128, out_features=1)

        self.relu = nn.ReLU()
        self.tanh = nn.Tanh()
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):

        h0 = torch.zeros(5, x.shape[0], 8)
        c0 = torch.zeros(5, x.shape[0], 8)

        x, hn = self.lstm(x, (h0,c0))

        x = x.reshape(x.size(0),-1)

        x = self.fc1(x)
        x = self.tanh(x)
        x = self.dropout(x)
        x = self.fc2(x)

        return x

In [16]:
model_L = torch.load('./model/LSTM_best.pth',weights_only=False)

In [17]:
S_number = ['126', '245', '370', '585']
for i in S_number:
    globals()['LSTM_result_SSP{}'.format(i)] = pd.DataFrame(columns=np.insert(Region,0,'date'))

In [18]:
data_SSP126['date']=pd.to_datetime(data_SSP126['date'])
data_SSP245['date']=pd.to_datetime(data_SSP245['date'])
data_SSP370['date']=pd.to_datetime(data_SSP370['date'])
data_SSP585['date']=pd.to_datetime(data_SSP585['date'])

In [19]:
LSTM_result_SSP126['date']=data_SSP126.loc[data_SSP126['date'].dt.strftime('%m-%d')>='05-22','date'].unique()
LSTM_result_SSP245['date']=data_SSP245.loc[data_SSP245['date'].dt.strftime('%m-%d')>='05-22','date'].unique()
LSTM_result_SSP370['date']=data_SSP370.loc[data_SSP370['date'].dt.strftime('%m-%d')>='05-22','date'].unique()
LSTM_result_SSP585['date']=data_SSP585.loc[data_SSP585['date'].dt.strftime('%m-%d')>='05-22','date'].unique()

In [20]:
LSTM_result_SSP126.index=LSTM_result_SSP126['date']
LSTM_result_SSP245.index=LSTM_result_SSP245['date']
LSTM_result_SSP370.index=LSTM_result_SSP370['date']
LSTM_result_SSP585.index=LSTM_result_SSP585['date']

In [ ]:
for i in S_number:
    for j in Region:
        for num, z in enumerate(range(2024,2073)):
            temp = globals()['data_SSP{}'.format(i)].loc[(globals()['data_SSP{}'.format(i)]['region']==j) & (globals()['data_SSP{}'.format(i)]['year']==z),:].copy()
            temp_feature=temp.loc[:,features]
            temp_X = (temp_feature-X_min)/(X_max-X_min)
            X_temp_w = make_dataset_X(temp_X, window_size)
            temp_pred_x = torch.Tensor(X_temp_w)
            temp_pred_y = model_L(temp_pred_x)
            globals()['LSTM_result_SSP{}'.format(i)].loc[temp['date'][2:],j]=temp_pred_y.detach().numpy()

In [22]:
LSTM_result_SSP126['year']=LSTM_result_SSP126['date'].dt.year
LSTM_result_SSP245['year']=LSTM_result_SSP245['date'].dt.year
LSTM_result_SSP370['year']=LSTM_result_SSP370['date'].dt.year
LSTM_result_SSP585['year']=LSTM_result_SSP585['date'].dt.year

In [23]:
for i in S_number:
    globals()['LSTM_result_SSP{}'.format(i)]['Korea'] = globals()['LSTM_result_SSP{}'.format(i)].loc[:, 'Gangwon':'Chungbuk'].sum(axis=1)

In [24]:
LSTM_result_SSP126.reset_index(drop=True, inplace=True)
LSTM_result_SSP245.reset_index(drop=True, inplace=True)
LSTM_result_SSP370.reset_index(drop=True, inplace=True)
LSTM_result_SSP585.reset_index(drop=True, inplace=True)

In [25]:
LSTM_result_SSP126

,date,Gangwon,Gyeonggi,Gyeongnam,Gyeongbuk,Gwangju,Daegu,Daejeon,Busan,Seoul,Ulsan,Incheon,Jeonnam,Jeonbuk,Jeju,Chungnam,Chungbuk,year,Korea
0,2024-05-22,0.109769,0.101588,0.182637,0.132578,0.166746,0.193139,0.149307,0.21131,0.167509,0.176483,0.1758,0.169258,0.134766,0.14544,0.102878,0.108413,2024,2.42762
1,2024-05-23,0.166167,0.226473,0.247123,0.361202,0.221557,0.269247,0.203738,0.225815,0.205357,0.2051,0.18156,0.216464,0.223495,0.147923,0.17958,0.19486,2024,3.47566
2,2024-05-24,0.235802,0.257851,0.261394,0.627459,0.227683,0.339347,0.255256,0.236401,0.250057,0.238858,0.197569,0.211134,0.243399,0.158638,0.198035,0.259059,2024,4.197943
3,2024-05-25,0.237476,0.409596,0.635138,0.953019,0.385802,0.537185,0.351327,0.313114,0.292531,0.319119,0.231001,0.436508,0.398156,0.159974,0.338439,0.41295,2024,6.411333
4,2024-05-26,0.152204,0.827739,0.249653,0.209632,0.288925,0.264383,0.290645,0.27437,0.436094,0.211936,0.339367,0.258619,0.345599,0.142481,0.482828,0.283384,2024,5.057859
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6463,2072-09-26,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2072,0
6464,2072-09-27,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2072,0
6465,2072-09-28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2072,0
6466,2072-09-29,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2072,0
